# Building footprints and use classification

I link OS building footprints to UPRNs, add building heights and classify their use. The residential subset feeds the completed dwelling stock.


In [ ]:
# Review copy: read the saved outputs without executing the research pipeline.
import os
if os.environ.get("ENERMAP_ENABLE_MODEL_RUN") != "1":
    raise RuntimeError("This showcase contains saved outputs. Raw data and third-party engines are excluded. Read the notebook; do not Run All for the presentation.")
import sys
from pathlib import Path
_repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.py").exists())
sys.path.insert(0, str(_repo))


## 1. Setup and boundary

In [ ]:
import os, time, warnings
from pathlib import Path

# The national OpenMap Local Buildings gpkg is "Measured 3D"; pyogrio warns on every read. Z/M are
# unused here, so silence that one warning.
warnings.filterwarnings("ignore", message="Measured .* not supported")

# GDAL/PROJ data dirs are not exported by this env's activation scripts -> set BEFORE importing pyogrio.
ENV = Path(os.environ.get("CONDA_PREFIX", r"LOCAL_PATH/pypsa"))
os.environ.setdefault("GDAL_DATA", str(ENV / "Library/share/gdal"))
os.environ.setdefault("PROJ_DATA", str(ENV / "Library/share/proj"))

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import shapely
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree

pd.set_option("display.max_columns", 80); pd.set_option("display.width", 230)
CRS = 27700
ROOT = Path(".").resolve()
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

LAD = gpd.read_file("gf_LADboundary/Guildford_LAD.shp").to_crs(CRS)
LAD_GEOM = LAD.geometry.union_all()
BBOX = tuple(LAD.total_bounds)
print(f"Guildford LAD: {LAD_GEOM.area/1e6:.0f} km2, bounds {[round(v) for v in BBOX]}")

## 2. Building footprints from MasterMap Topography

Read the `Building` polygons from all 16 tiles (filter pushed down to GDAL), de-duplicate on TOID
(tiles overlap), and clip to the boundary by intersection (whole footprints kept).

In [ ]:
t0 = time.time()
parts = [gpd.read_file(t, layer="Topographicarea", engine="pyogrio",
                       where="descriptivegroup LIKE '%Building%'", bbox=BBOX)
         for t in sorted(Path("Guildford Os topography").glob("*/*.gpkg"))]
bld = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), geometry="geometry", crs=CRS)
bld = bld.rename(columns={"fid": "toid"}).drop_duplicates("toid")
bld = bld[bld.intersects(LAD_GEOM)].reset_index(drop=True)
bld["area_m2"] = shapely.area(bld.geometry.values)
n = len(bld)
print(f"{n:,} building footprints ({time.time()-t0:.0f}s), {bld.area_m2.sum()/1e4:,.0f} ha")
bld[["toid", "descriptivegroup", "descriptiveterm", "area_m2"]].head(3)

## 3. Building heights (Building Height Attribute)

In [ ]:
HCOLS = ["os_topo_toid", "abshmin", "absh2", "abshmax", "relh2", "relhmax", "bha_conf"]
t0 = time.time()
hgt = pd.concat([pyogrio.read_dataframe(g, layer=pyogrio.list_layers(g)[0][0],
                                        read_geometry=False, columns=HCOLS)
                 for g in Path("Guildford OS heights").glob("*/*/*.gdb")], ignore_index=True)
hgt = hgt.drop_duplicates("os_topo_toid")
bld = bld.merge(hgt, left_on="toid", right_on="os_topo_toid", how="left").drop(columns="os_topo_toid")

bld["has_height"]   = bld.relhmax.notna()
bld["height_eaves_m"] = bld.relh2
bld["height_ridge_m"] = bld.relhmax
bld["ground_level_m"] = bld.abshmin
bld["is_flat_roof"]   = (bld.relhmax - bld.relh2).clip(lower=0) < 0.5
bld["storeys_est"]    = np.maximum(1, (bld.relh2 / 3.0).round()).where(bld.has_height)
bld["volume_m3"]      = (bld.area_m2 * bld.relh2).round(1)
print(f"heights joined ({time.time()-t0:.0f}s): {bld.has_height.sum():,}/{n:,} matched "
      f"({100*bld.has_height.mean():.1f}%), median ridge {bld.height_ridge_m.median():.1f} m")

## 4. Link footprints to UPRNs

The OS address cross-reference links each UPRN to a building TOID. I leave out footprints with no linked address, such as most garages and sheds.


In [ ]:
t0 = time.time()
# OS BLPU UPRN <-> TopographicArea TOID cross-reference (authoritative, AddressBase-derived).
# National 6.2 GB CSV: IDENTIFIER_1 = UPRN, IDENTIFIER_2 = TOID. Stream it filtered to our TOIDs.
BLPU = next(Path("TOID_UPRN link").glob("BLPU_UPRN_TopographicArea_TOID*.csv"))
tset = set(bld.toid)
parts = []
for ch in pd.read_csv(BLPU, usecols=["IDENTIFIER_1", "IDENTIFIER_2"], dtype=str, chunksize=4_000_000):
    m = ch[ch.IDENTIFIER_2.isin(tset)]
    if len(m):
        parts.append(m)
link = pd.concat(parts, ignore_index=True).rename(columns={"IDENTIFIER_1": "UPRN", "IDENTIFIER_2": "toid"})
link["UPRN"] = link.UPRN.astype("int64")

# Roll up to the building: uprn_list is the full set (for flats), uprn_primary the lowest (1:1 key).
grp = link.groupby("toid").UPRN
bld["uprn_count"] = bld.toid.map(grp.size()).fillna(0).astype(int)
bld["has_uprn"] = bld.uprn_count > 0
bld["uprn_list"] = bld.toid.map(grp.apply(lambda s: ";".join(map(str, sorted(s))))).fillna("")
bld["uprn_primary"] = bld.toid.map(grp.min()).fillna(0).astype("int64")
print(f"BLPU link ({BLPU.stat().st_size/1e9:.1f} GB) streamed+filtered ({time.time()-t0:.0f}s): "
      f"{len(link):,} UPRN->TOID links | addressed footprints {int(bld.has_uprn.sum()):,}")

# --- Drop unaddressed structures (no UPRN) BEFORE adjacency / analysis ---
# Buildings with no UPRN are overwhelmingly garages, sheds and attachments (median ~16 m2). They
# distort party-wall neighbour counts (a house + its garage looks semi-detached), so they are
# removed up front and neighbours are counted only among real, addressed buildings. NB a small
# number of evidenced/utility structures without a UPRN are also dropped - recover from raw data if
# non-residential completeness matters.
before = len(bld)
dropped = bld[~bld.has_uprn]
bld = bld[bld.has_uprn].reset_index(drop=True)
lookup = link[["UPRN", "toid"]].reset_index(drop=True)   # all links land on kept buildings
n = len(bld)
print(f"dropped {before-n:,} unaddressed footprints (no UPRN, median {dropped.area_m2.median():.0f} m2, "
      f"{100*(dropped.area_m2<30).mean():.0f}% <30 m2) -> {n:,} addressed buildings remain")
print(f"  buildings holding >1 UPRN (flats/multi-occupancy): {int((bld.uprn_count>1).sum()):,}")

## 5. Adjacency blocks and built form

Party-wall adjacency (shared boundary ≥ 1 m) → connected blocks → morphological `built_form`,
computed on the **addressed buildings only** (§4), so a house attached solely to its dropped garage
now reads correctly as detached. Used later to propagate evidence and to name house types.

In [ ]:
t0 = time.time()
geom = bld.geometry.values
pairs = gpd.sjoin(bld[["geometry"]], bld[["geometry"]], how="inner", predicate="touches")
li, ri = pairs.index.values, pairs.index_right.values
keep = shapely.length(shapely.intersection(geom[li], geom[ri])) >= 1.0
li, ri = li[keep], ri[keep]
bld["n_neighbours"] = pd.Series(li).value_counts().reindex(range(n), fill_value=0).values
_, block = connected_components(coo_matrix((np.ones(len(li)), (li, ri)), shape=(n, n)), directed=False)
bld["block_id"] = block
bld["block_size"] = pd.Series(block).map(pd.Series(block).value_counts()).values
bld["built_form"] = np.select(
    [bld.block_size == 1, bld.block_size == 2, bld.n_neighbours == 1],
    ["Detached", "Semi-detached", "End-terrace"], default="Mid-terrace")
cen = bld.geometry.centroid; bld["cen_x"], bld["cen_y"] = cen.x.values, cen.y.values
print(f"adjacency ({time.time()-t0:.0f}s): {bld.block_id.nunique():,} blocks, "
      f"largest {bld.block_size.max()}")

## 6. Evidence — MasterMap `descriptiveterm` and house-number labels

`descriptiveterm` flags special structures (substations etc.); `Cartographictext` numeric labels
inside a footprint flag addressed dwellings.

In [ ]:
term = bld.descriptiveterm.fillna("")
bld["is_special_structure"] = term.str.strip().ne("")
bld["is_substation"] = term.str.contains("Electricity Sub Station", regex=False)

txt = pd.concat([gpd.read_file(t, layer="Cartographictext", engine="pyogrio",
                               bbox=BBOX, columns=["textstring"])
                 for t in sorted(Path("Guildford Os topography").glob("*/*.gpkg"))], ignore_index=True)
txt = gpd.GeoDataFrame(txt, geometry="geometry", crs=CRS).drop_duplicates()
lbl = gpd.sjoin(txt, bld[["toid", "geometry"]], how="inner", predicate="within")
num = lbl.textstring.fillna("").str.match(r"^\d+[A-Za-z]?$")
rng = lbl.textstring.fillna("").str.contains(r"\d+\s+to\s+\d+")
bld["has_house_number"] = bld.toid.isin(set(lbl.loc[num | rng, "toid"]))
print(f"special structures: {int(bld.is_special_structure.sum()):,} "
      f"(substations {int(bld.is_substation.sum())}) | house numbers: {int(bld.has_house_number.sum()):,}")

## 7. Evidence — OS Points of Interest (premises)

Clip both POI squares, split **premises** from street **furniture** (bus stops, cash machines…),
flag **soft** premises (Commercial Services — home-registered businesses), and link to footprints by
the exact `topo_toid` key ∪ spatial containment.

In [ ]:
NON_PREMISES_CAT = {"19", "58", "55", "59", "54"}
FURN_KW = ("electrical feature|gas feature|meteorological|telecommunications feature|letter box|"
           "drinking fountain|wifi hotspot|trigonometric|cash machine|footbridge|playground|"
           "pipeline|taxi rank|bus stop|car park|parking|public telephone|recycling centre|"
           "picnic|street furniture")
poi = pd.concat([gpd.read_file(f, engine="pyogrio", bbox=BBOX,
                               columns=["pointx_class", "classname", "groupname", "name", "topo_toid"])
                 for f in sorted(Path("Guildford OS poi").glob("*/*.gpkg"))], ignore_index=True)
poi = gpd.GeoDataFrame(poi, geometry="geometry", crs=CRS)
poi = poi[poi.within(LAD_GEOM)].drop_duplicates(["name", "topo_toid", "pointx_class"])
poi["cat"] = poi.pointx_class.str[2:4]; poi["grp"] = poi.pointx_class.str[:2]
furniture = poi.cat.isin(NON_PREMISES_CAT) | poi.classname.str.lower().str.contains(FURN_KW, regex=True, na=False)
prem = poi[~furniture].copy()
prem["is_soft"] = prem.grp == "02"

by_key = prem[prem.topo_toid.isin(set(bld.toid))].assign(toid=lambda d: d.topo_toid)
by_geom = gpd.sjoin(prem, bld[["toid", "geometry"]], how="inner", predicate="within")
links = pd.concat([by_key[["toid", "name", "is_soft"]], by_geom[["toid", "name", "is_soft"]]]).drop_duplicates(["toid", "name"])
roll = links.groupby("toid").is_soft.agg(poi_count="size", poi_hard_count=lambda s: int((~s).sum()))
bld = bld.merge(roll, on="toid", how="left")
bld["poi_count"] = bld.poi_count.fillna(0).astype(int)
bld["poi_hard_count"] = bld.poi_hard_count.fillna(0).astype(int)
bld["poi_soft_only"] = (bld.poi_count > 0) & (bld.poi_hard_count == 0)
print(f"POI premises: {len(prem):,} | footprints with premises {int((bld.poi_count>0).sum()):,} "
      f"(hard {int((bld.poi_hard_count>0).sum()):,})")

## 8. Evidence — MasterMap Sites and OpenMap Local ImportantBuilding

MasterMap functional sites (incl. utility/rail); dissolve multi-part sites by TOID. OpenMap Local
ImportantBuilding gives per-building institutional use.

In [ ]:
rep = bld[["toid"]].copy(); rep["geometry"] = bld.geometry.representative_point()
rep = gpd.GeoDataFrame(rep, geometry="geometry", crs=CRS)

# MasterMap Sites
sraw = pd.concat([gpd.read_file(f, layer="Functional Sites", engine="pyogrio", bbox=BBOX)
                  for f in sorted(Path("Guildford OS sites").glob("*/*.gpkg"))], ignore_index=True)
sites = gpd.GeoDataFrame(sraw, geometry="geometry", crs=CRS).dissolve(by="toid", aggfunc="first").reset_index()
sites = sites[sites.intersects(LAD_GEOM)]
sj = gpd.sjoin(rep, sites[["functiontheme", "geometry"]], how="inner", predicate="within").drop_duplicates("toid")
bld = bld.merge(sj[["toid", "functiontheme"]].rename(columns={"functiontheme": "site_theme"}), on="toid", how="left")

# OpenMap Local ImportantBuilding (SU + TQ shapefiles)
OSL = Path("Guildford OS local")
imp = pd.concat([gpd.read_file(OSL / f"opmplc_essh_{t}/OS OpenMap Local (ESRI Shape File) {t.upper()}/data/{t.upper()}_ImportantBuilding.shp",
                               engine="pyogrio", bbox=BBOX) for t in ("su", "tq")], ignore_index=True)
imp = gpd.GeoDataFrame(imp, geometry="geometry", crs=CRS).drop_duplicates("ID")
ov = gpd.overlay(bld[["toid", "geometry"]], imp[["BUILDGTHEM", "geometry"]], how="intersection", keep_geom_type=True)
ov["frac"] = ov.geometry.area / ov.toid.map(bld.set_index("toid").area_m2)
best = ov[ov.frac >= 0.5].sort_values("frac").drop_duplicates("toid", keep="last")
bld = bld.merge(best[["toid", "BUILDGTHEM"]].rename(columns={"BUILDGTHEM": "imp_theme"}), on="toid", how="left")

# OpenMap Local generalised Building blocks (national gpkg, indexed) -> ancillary signal
osb = gpd.read_file(next(Path("Guildford OS buildings").glob("*.gpkg")), engine="pyogrio", bbox=BBOX)
sjb = gpd.sjoin(rep, osb[["geometry"]], how="left", predicate="within").drop_duplicates("toid")
bld["in_osl_block"] = bld.toid.isin(set(sjb.loc[sjb.index_right.notna(), "toid"]))
print(f"in a MasterMap site: {int(bld.site_theme.notna().sum()):,} | ImportantBuilding: "
      f"{int(bld.imp_theme.notna().sum()):,} | in OS Local block: {int(bld.in_osl_block.sum()):,}")

## 9. Use classification — evidence tiers

Highest priority first; surveyed sources beat business listings; the soft-POI / home-business
conflict is resolved so a house with a registered consultancy stays residential. Adjacency
propagation (tiers 8–9) spreads sparse evidence across attached blocks.

In [ ]:
def propagate(mask):
    return bld.block_id.isin(set(bld.block_id[mask.values].unique()))

t1 = bld.is_special_structure
t2 = bld.site_theme.notna()
t3 = bld.imp_theme.notna()
t4 = bld.poi_hard_count > 0
t5 = bld.poi_soft_only & bld.has_house_number
t6 = bld.poi_soft_only & ~bld.has_house_number
t7 = bld.has_house_number
prior = t1 | t2 | t3 | t4 | t5 | t6 | t7
t8 = propagate(t2 | t3 | t4 | t6) & ~prior
t9 = propagate(t5 | t7) & ~prior & ~t8
# (No geometry/size Ancillary tier: the shed/garage population was dropped in §4, so every
#  remaining building is a real addressed structure.)

bld["use_class"] = "Unknown"; bld["use_confidence"] = "none"; bld["use_evidence"] = "none"
for mask, cls, conf, ev in [
    (t9, "Residential", "medium", "block propagated (house number)"),
    (t8, "Non-residential", "medium", "block propagated"),
    (t7, "Residential", "medium", "house number label"),
    (t6, "Non-residential", "medium", "POI premises (soft, unaddressed)"),
    (t5, "Residential", "medium", "house number + home business"),
    (t4, "Non-residential", "high", "POI premises"),
    (t3, "Non-residential", "high", "OS Local ImportantBuilding"),
    (t2, "Non-residential", "high", "MasterMap functional site"),
    (t1, "Utility structure", "high", "MasterMap descriptiveterm"),
]:
    bld.loc[mask, ["use_class", "use_confidence", "use_evidence"]] = [cls, conf, ev]
bld["is_home_business"] = t5

U0 = int((bld.use_class == "Unknown").sum())
print(f"evidence classification: Unknown {U0:,} ({100*U0/n:.1f}%)")
bld.use_class.value_counts().to_frame("footprints").assign(pct=lambda d: (100*d.footprints/n).round(1))

## 10. Resolve the Unknowns — UPRN address → neighbour context

Two fills, written to `use_class_final` / `confidence_final` / `fill_method` so the evidence
classification (§9) stays pristine and every fill is filterable. Every building here is addressed,
so address evidence does most of the work; a neighbour vote mops up the few large evidence-poor ones.

In [ ]:
bld["use_class_final"] = bld.use_class
bld["confidence_final"] = bld.use_confidence
bld["fill_method"] = np.where(bld.use_class == "Unknown", "none", "evidence")
bld["context_dist_m"] = np.nan

# Tier 1 - UPRN address evidence: an addressed building with no commercial evidence is a dwelling;
# >=3 stacked UPRNs is a flats block. (No geometry Ancillary tier - unaddressed structures gone.)
uk = bld.use_class_final == "Unknown"
mF = uk & (bld.uprn_count >= 3)
mD = uk & bld.uprn_count.isin([1, 2]) & (bld.area_m2 <= 400)
bld.loc[mF, ["use_class_final", "confidence_final", "fill_method"]] = ["Residential", "medium", "UPRN (multi-dwelling)"]
bld.loc[mD, ["use_class_final", "confidence_final", "fill_method"]] = ["Residential", "medium", "UPRN (addressed dwelling)"]

# Tier 2 - neighbour-majority vote among originally-confident buildings (single pass), for the few
# large addressed buildings with no other evidence.
CONF = bld.use_confidence.isin(["high", "medium"]) & bld.use_class.isin(["Residential", "Non-residential"])
vc = np.c_[bld[CONF].cen_x.values, bld[CONF].cen_y.values]
vlab = (bld[CONF].use_class.values == "Residential").astype(int)
tree = cKDTree(vc)
pool = bld[bld.use_class_final == "Unknown"]; pts = np.c_[pool.cen_x.values, pool.cen_y.values]
idx = tree.query_ball_point(pts, r=50.0); assigned = 0
for row_i, nb_ in zip(pool.index, idx):
    if len(nb_) < 3: continue
    share = vlab[nb_].mean(); purity = max(share, 1 - share)
    if purity < 0.7: continue
    d = float(np.hypot(bld.at[row_i, "cen_x"] - vc[nb_, 0], bld.at[row_i, "cen_y"] - vc[nb_, 1]).min())
    cls = "Residential" if share >= 0.5 else "Non-residential"
    conf = "medium" if (d <= 30 and cls == "Residential") else "low"
    bld.loc[row_i, ["use_class_final", "confidence_final", "fill_method", "context_dist_m"]] = [cls, conf, "neighbour-majority (spatial context)", round(d, 1)]
    assigned += 1

uf = int((bld.use_class_final == "Unknown").sum())
print(f"Tier1 UPRN {int((mF|mD).sum()):,} | Tier2 context {assigned:,}")
print(f"Unknown: {U0:,} ({100*U0/n:.1f}%)  ->  {uf:,} ({100*uf/n:.1f}%)")

## 11. Flats — `dwelling_type`

Footprint shape can't see flats (they stack vertically); UPRN counts can. A residential footprint
with **≥ 3 stacked UPRNs → Flats**, else its house form from `built_form`.

In [ ]:
HOUSE = {"Detached": "Detached house", "Semi-detached": "Semi-detached house",
         "End-terrace": "Terraced house", "Mid-terrace": "Terraced house"}
bld["dwelling_type"] = np.where(
    bld.use_class_final != "Residential", "",
    np.where(bld.uprn_count >= 3, "Flats", bld.built_form.map(HOUSE).fillna("House")))

resid = bld[bld.use_class_final == "Residential"]
print(f"{len(resid):,} residential | Flats {int((bld.dwelling_type=='Flats').sum()):,} blocks "
      f"holding {int(bld.loc[bld.dwelling_type=='Flats','uprn_count'].sum()):,} dwellings")
display(resid.dwelling_type.value_counts().to_frame("buildings"))

## 12. Summary

In [ ]:
order = ["Residential", "Non-residential", "Ancillary", "Utility structure", "Unknown"]
print(pd.DataFrame({"rule_only": bld.use_class.value_counts(),
                    "final": bld.use_class_final.value_counts()}).fillna(0).astype(int)
      .reindex(order).assign(final_pct=lambda d: (100*d.final/n).round(1)).to_string())
print("\nlabel provenance (fill_method):")
print(bld.fill_method.value_counts().to_string())

cmap = {"Residential": "#55A868", "Non-residential": "#C44E52", "Ancillary": "#CCB974",
        "Utility structure": "#8172B2", "Unknown": "#BBBBBB"}
fig, ax = plt.subplots(1, 2, figsize=(15, 4.4))
for a, col, ti in [(ax[0], "use_class", "Evidence only"), (ax[1], "use_class_final", "Final")]:
    bld[col].value_counts().reindex(order).plot.bar(ax=a, color=[cmap[c] for c in order])
    a.set_title(f"{ti} — Unknown {100*(bld[col]=='Unknown').mean():.1f}%"); a.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

## 13. Interactive map — footprints on a basemap (leafmap)

In [ ]:
import leafmap.foliumap as leafmap
DCOL = {"Detached house": "#4C72B0", "Semi-detached house": "#55A868", "Terraced house": "#DD8452",
        "Flats": "#C44E52", "House": "#7aa8d6", "Non-residential": "#8c3b3b",
        "Ancillary": "#CCB974", "Utility structure": "#8172B2", "Unknown": "#BBBBBB"}
bld["display_class"] = np.where(bld.dwelling_type != "", bld.dwelling_type, bld.use_class_final)
bld["fillColor"] = bld.display_class.map(DCOL).fillna("#BBBBBB")

cx0, cy0, H = 499300, 149600, 800
win = bld.cx[cx0-H:cx0+H, cy0-H:cy0+H][
    ["toid", "display_class", "use_class_final", "confidence_final", "dwelling_type",
     "storeys_est", "height_ridge_m", "uprn_count", "fillColor", "geometry"]].to_crs(4326)
c = win.geometry.union_all().centroid
m = leafmap.Map(center=[c.y, c.x], zoom=16)
m.add_basemap("CartoDB.Positron")
m.add_gdf(win, layer_name=f"Footprints ({len(win):,})",
          style_function=lambda f: {"fillColor": f["properties"]["fillColor"], "color": "white",
                                    "weight": 0.4, "fillOpacity": 0.75}, info_mode="on_hover")
m.add_legend(title="Building class", legend_dict={k: v for k, v in DCOL.items() if k in set(win.display_class)})
m

## 14. Export — the single outcome

In [ ]:
KEEP = ["toid", "descriptivegroup", "descriptiveterm", "area_m2",
        "height_eaves_m", "height_ridge_m", "storeys_est", "volume_m3", "is_flat_roof",
        "ground_level_m", "bha_conf", "has_height",
        "n_neighbours", "block_id", "block_size", "built_form",
        "has_house_number", "poi_count", "poi_hard_count", "poi_soft_only",
        "site_theme", "imp_theme", "in_osl_block",
        "uprn_count", "has_uprn", "uprn_primary", "uprn_list",
        "is_substation", "is_special_structure", "is_home_business",
        "use_class", "use_confidence", "use_evidence",
        "use_class_final", "confidence_final", "fill_method", "context_dist_m", "dwelling_type"]
result = bld[KEEP + ["geometry"]].copy()

gpkg = OUT / "guildford_buildings.gpkg"
if gpkg.exists(): gpkg.unlink()
result.to_file(gpkg, layer="building", driver="GPKG")
result.drop(columns="geometry").to_csv(OUT / "guildford_buildings.csv", index=False)
lookup.to_csv(OUT / "uprn_toid_lookup.csv", index=False)

info = pyogrio.read_info(gpkg, layer="building")
print(f"OUTCOME -> {gpkg.resolve()}")
print(f"  {info['features']:,} building footprints x {len(KEEP)} attributes, {gpkg.stat().st_size/1e6:.0f} MB")
print(f"  + guildford_buildings.csv, uprn_toid_lookup.csv ({len(lookup):,} rows)")

## 15. Residential subset

A second GeoPackage of the **dwelling stock** — `Residential` + `Unknown` footprints whose
**area is between 12 and 800 m²** — for dwelling-level work (UBEM, EPC linkage, rooftop PV).

- **Class filter:** keeps `Residential` and `Unknown`; drops `Non-residential` and `Utility structure`.
  (Garages/sheds are already gone — they were dropped as unaddressed in §4.)
- **Area filter:** `12 ≤ area_m2 ≤ 800`. Removes sub-12 m² slivers and >800 m² footprints (large
  blocks / mis-merged geometry that aren't single dwellings).

Adjust `SUBSET` or `AREA_MIN` / `AREA_MAX` below to change either filter.

In [ ]:
SUBSET = ["Residential", "Unknown"]
AREA_MIN, AREA_MAX = 12, 800          # m2 - keep only plausibly-dwelling-sized footprints

subset0 = result[result.use_class_final.isin(SUBSET)]
res = subset0[subset0.area_m2.between(AREA_MIN, AREA_MAX)].copy()

rgpkg = OUT / "guildford_residential_footprint.gpkg"
if rgpkg.exists(): rgpkg.unlink()
res.to_file(rgpkg, layer="building", driver="GPKG")
res.drop(columns="geometry").to_csv(OUT / "guildford_residential_footprint.csv", index=False)

print(f"{rgpkg.name} -> {len(res):,} footprints ({100*len(res)/n:.1f}% of all buildings), "
      f"{rgpkg.stat().st_size/1e6:.0f} MB")
print(f"  filter: use_class_final in {SUBSET}  AND  {AREA_MIN} <= area_m2 <= {AREA_MAX}")
print(f"  area filter removed {len(subset0)-len(res):,} "
      f"(< {AREA_MIN} m2: {int((subset0.area_m2 < AREA_MIN).sum()):,}, "
      f"> {AREA_MAX} m2: {int((subset0.area_m2 > AREA_MAX).sum()):,})")
print(f"  kept    : {', '.join(f'{k} {int(v):,}' for k, v in res.use_class_final.value_counts().items())}")
print(f"  excluded: {', '.join(f'{k} {int(v):,}' for k, v in result.loc[~result.use_class_final.isin(SUBSET), 'use_class_final'].value_counts().items())}")
print(f"  dwellings addressed in the subset: {int(res.uprn_count.sum()):,} UPRNs")
display(res.dwelling_type.value_counts().to_frame("footprints"))

## 16. Slim exports for analysis (`export/` folder)

Trimmed copies of both files with only the columns needed downstream — id, footprint area, height
metrics, roof/form, dwelling type, a **binary** `use_class_final` (Residential / Non-residential),
and the UPRNs. All the evidence/working columns used only to *build* the classification are dropped.

`use_class_final` is collapsed to two values: the unresolved **Unknown** buildings become
**Residential** (they are addressed dwellings we simply could not confirm), and Utility structures
join **Non-residential** — so `export/guildford_residential.gpkg` is 100 % Residential.

In [ ]:
EXPORT = Path("export"); EXPORT.mkdir(exist_ok=True)

# Slim column set for downstream analysis - id, geometry/physical metrics, form, class, addresses.
# The evidence/working columns (poi_*, site_*, use_class, confidence_*, fill_method, block ids, …)
# are dropped: they were only needed to *build* the classification, not to use it.
SLIM = ["toid", "area_m2",
        "height_eaves_m", "height_ridge_m", "storeys_est", "volume_m3", "is_flat_roof",
        "built_form", "dwelling_type", "use_class_final",
        "uprn_count", "uprn_primary", "uprn_list"]

# Collapse use class to a clean residential / non-residential binary. Addressed buildings we could
# not resolve (Unknown) are treated as Residential; substations/utility count as Non-residential.
BINARY = {"Residential": "Residential", "Unknown": "Residential",
          "Non-residential": "Non-residential", "Utility structure": "Non-residential"}

def slim(gdf):
    out = gdf[SLIM + ["geometry"]].copy()
    out["use_class_final"] = out.use_class_final.map(BINARY)
    return out

for name, src in [("guildford_buildings", result), ("guildford_residential", res)]:
    g = slim(src)
    p = EXPORT / f"{name}.gpkg"
    if p.exists(): p.unlink()
    g.to_file(p, layer="building", driver="GPKG")
    g.drop(columns="geometry").to_csv(EXPORT / f"{name}.csv", index=False)
    print(f"export/{name}.gpkg -> {len(g):,} rows x {len(SLIM)} cols "
          f"| use_class_final: {dict(g.use_class_final.value_counts())}")

print("\ncolumns kept:", SLIM)

### Outputs

- `guildford_buildings.gpkg`: addressed footprints with height, use class and linked UPRNs.
- `guildford_residential_footprint.gpkg`: the residential subset.
- `uprn_toid_lookup.csv`: one UPRN-to-building link per address, used to join EPC records.
